In [1]:
import os
import pandas as pd
import numpy as np
from speedml import Speedml

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
# Define data path
DATA_PATH = os.path.join('../../data/raw/', 'raw_test_call_data.csv')


In [14]:
# Initialize Speedml with the dataset
# Speedml requires a 'test' positional argument; set to None if no separate test set
sml = Speedml(train=DATA_PATH, test=None, target=None)

print(f"Dataset loaded successfully!")
print(f"Shape: {sml.train.shape}")

ValueError: Invalid file path or buffer object type: <class 'NoneType'>

## Initial Data Exploration

In [6]:
# Statistical summary of numerical features
sml.train.describe()

NameError: name 'sml' is not defined

## Data Quality Analysis

In [ ]:
# Data types summary
print("Data Types:")
print(sml.train.dtypes.value_counts())
print("\n" + "="*50 + "\n")

# Identify numerical and categorical columns
numerical_cols = sml.train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = sml.train.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")

## Visualizations

In [ ]:
# Box plots for numerical columns (to identify outliers)
if len(numerical_cols) > 0:
    n_cols = min(3, len(numerical_cols))
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if len(numerical_cols) > 1 else [axes]
    
    for idx, col in enumerate(numerical_cols):
        if idx < len(axes):
            sns.boxplot(y=sml.train[col], ax=axes[idx])
            axes[idx].set_title(f'Box Plot of {col}')
            axes[idx].set_ylabel(col)
    
    # Hide unused subplots
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns to plot.")

In [ ]:
# Bar plots for categorical columns (top categories)
if len(categorical_cols) > 0:
    for col in categorical_cols[:5]:  # Limit to first 5 categorical columns
        plt.figure(figsize=(12, 6))
        
        # Get value counts
        value_counts = sml.train[col].value_counts().head(20)  # Top 20 categories
        
        # Create bar plot
        value_counts.plot(kind='bar', edgecolor='black')
        plt.title(f'Distribution of {col} (Top 20 Categories)')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        
        print(f"\nValue counts for {col}:")
        print(value_counts)
        print("\n" + "="*50 + "\n")
else:
    print("No categorical columns to plot.")

In [ ]:
# Statistical outlier detection using IQR method
print("Outlier Analysis (IQR Method):")
print("="*60)

for col in numerical_cols:
    Q1 = sml.train[col].quantile(0.25)
    Q3 = sml.train[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = sml.train[(sml.train[col] < lower_bound) | (sml.train[col] > upper_bound)]
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(sml.train)) * 100
    
    print(f"\n{col}:")
    print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"  Lower bound: {lower_bound:.2f}, Upper bound: {upper_bound:.2f}")
    print(f"  Outliers: {outlier_count} ({outlier_percentage:.2f}%)")

In [ ]:
# Generate comprehensive summary
print("="*70)
print("EXPLORATORY DATA ANALYSIS SUMMARY")
print("="*70)

print(f"\nDataset Shape: {sml.train.shape[0]} rows × {sml.train.shape[1]} columns")

print(f"\nColumn Types:")
print(f"  - Numerical: {len(numerical_cols)}")
print(f"  - Categorical: {len(categorical_cols)}")

print(f"\nData Quality:")
missing_cols = missing_data[missing_data['Missing_Count'] > 0]
print(f"  - Columns with missing values: {len(missing_cols)}")
print(f"  - Duplicate rows: {duplicate_count}")

print(f"\nMemory Usage:")
print(f"  - Total: {sml.train.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "="*70)
print("\nNext Steps for Analysis:")
print("  1. Handle missing values if necessary")
print("  2. Investigate and treat outliers")
print("  3. Engineer new features based on domain knowledge")
print("  4. Consider encoding categorical variables")
print("  5. Perform feature scaling if needed for modeling")
print("="*70)

## Using Speedml for Further Analysis

The `sml` object provides additional methods for data preprocessing and feature engineering:

- **Data Cleaning**: `sml.feature.drop()`, `sml.feature.fillna()`
- **Feature Engineering**: `sml.feature.extract()`, `sml.feature.add()`
- **Data Transformation**: `sml.feature.density()`, `sml.feature.log()`
- **Encoding**: `sml.feature.mapping()`, `sml.feature.labels()`
- **Outlier Treatment**: `sml.feature.outliers()`

Access the dataframe directly via `sml.train` for custom operations.

## Summary & Key Insights

In [ ]:
# Use Speedml's feature correlation with target
# Note: This is useful if you have a target variable
# For now, we'll show pairwise correlations

print("Top Correlations Between Features:")
if len(numerical_cols) > 1:
    corr_matrix = sml.train[numerical_cols].corr()
    
    # Get the correlation pairs
    corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })
    
    corr_df = pd.DataFrame(corr_pairs).sort_values('Correlation', 
                                                     key=abs, 
                                                     ascending=False)
    print(corr_df.head(10))
else:
    print("Need at least 2 numerical columns for correlation analysis.")

## Speedml-Specific Analysis

In [ ]:
# Correlation heatmap for numerical columns
if len(numerical_cols) > 1:
    plt.figure(figsize=(10, 8))
    correlation_matrix = sml.train[numerical_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1)
    plt.title('Correlation Matrix of Numerical Features')
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 numerical columns for correlation analysis.")

In [ ]:
# Distribution plots for numerical columns
if len(numerical_cols) > 0:
    n_cols = min(3, len(numerical_cols))
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if len(numerical_cols) > 1 else [axes]
    
    for idx, col in enumerate(numerical_cols):
        if idx < len(axes):
            sml.train[col].hist(bins=30, ax=axes[idx], edgecolor='black')
            axes[idx].set_title(f'Distribution of {col}')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Frequency')
    
    # Hide unused subplots
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns to plot.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Cardinality analysis for categorical columns
print("Cardinality Analysis (Unique Values per Categorical Column):")
for col in categorical_cols:
    unique_count = sml.train[col].nunique()
    print(f"{col}: {unique_count} unique values")
    if unique_count <= 10:
        print(f"  Values: {sml.train[col].unique()}")
    print()

In [ ]:
# Check for duplicate rows
duplicate_count = sml.train.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")
print(f"Percentage of duplicates: {(duplicate_count / len(sml.train)) * 100:.2f}%")

In [ ]:
# Check for missing values
null_counts = sml.train.isnull().sum()
null_percentages = (null_counts / len(sml.train)) * 100

missing_data = pd.DataFrame({
    'Column': null_counts.index,
    'Missing_Count': null_counts.values,
    'Missing_Percentage': null_percentages.values
}).sort_values('Missing_Count', ascending=False)

print("Missing Values Summary:")
missing_data[missing_data['Missing_Count'] > 0]

In [ ]:
# Get dataset information
sml.train.info()

In [ ]:
# Display first few rows
sml.train.head()